Imports and load

In [8]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
df = pd.read_csv(r"C:\Users\USER\Documents\MIT8339_BI_Virtual_Lab\data\BankChurners.csv")
print(pd.__version__)
df.head()

2.2.2


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [10]:
# Display the first five rows
print (df.shape)

# Check column names and data types
df.info()

(10127, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  object 
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                               

Find missing values and duplicates

In [14]:
# Check for missing values
print("Missing values per column:\n", df.isnull().sum())

# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())
print("CLIENTNUM unique:", df["CLIENTNUM"].is_unique)

Missing values per column:
 CLIENTNUM                                                                                                                             0
Attrition_Flag                                                                                                                        0
Customer_Age                                                                                                                          0
Gender                                                                                                                                0
Dependent_count                                                                                                                       0
Education_Level                                                                                                                       0
Marital_Status                                                                                                                        0
Income_Category     

Checking for inconsistent labels

In [16]:
cat_cols = ["Attrition_Flag", "Gender", "Education_Level", "Marital_Status",
            "Income_Category", "Card_Category"]
for c in cat_cols:
    print(df[c].value_counts(), "\n")

Attrition_Flag
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64 

Gender
F    5358
M    4769
Name: count, dtype: int64 

Education_Level
Graduate         3128
High School      2013
Unknown          1519
Uneducated       1487
College          1013
Post-Graduate     516
Doctorate         451
Name: count, dtype: int64 

Marital_Status
Married     4687
Single      3943
Unknown      749
Divorced     748
Name: count, dtype: int64 

Income_Category
Less than $40K    3561
$40K - $60K       1790
$80K - $120K      1535
$60K - $80K       1402
Unknown           1112
$120K +            727
Name: count, dtype: int64 

Card_Category
Blue        9436
Silver       555
Gold         116
Platinum      20
Name: count, dtype: int64 



The validity checks

In [18]:
print("Age outside 18-100:", ((df["Customer_Age"] < 18) | (df["Customer_Age"] > 100)).sum())
print("Utilisation outside 0-1:", ((df["Avg_Utilization_Ratio"] < 0) | (df["Avg_Utilization_Ratio"] > 1)).sum())
print("Non-positive trans count:", (df["Total_Trans_Ct"] <= 0).sum())
print("Non-positive trans amount:", (df["Total_Trans_Amt"] <= 0).sum())
diff = (df["Credit_Limit"] - df["Total_Revolving_Bal"] - df["Avg_Open_To_Buy"]).abs()
print("Max open-to-buy mismatch:", diff.max())

Age outside 18-100: 0
Utilisation outside 0-1: 0
Non-positive trans count: 0
Non-positive trans amount: 0
Max open-to-buy mismatch: 5.684341886080802e-14


Outliers (IQR)

In [20]:
num_cols = ["Customer_Age", "Credit_Limit", "Total_Revolving_Bal",
            "Total_Trans_Amt", "Total_Trans_Ct", "Avg_Utilization_Ratio"]
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    print(f"{c}: {n} outliers")

Customer_Age: 2 outliers
Credit_Limit: 984 outliers
Total_Revolving_Bal: 0 outliers
Total_Trans_Amt: 896 outliers
Total_Trans_Ct: 2 outliers
Avg_Utilization_Ratio: 0 outliers


Clean the dataset: "Unknown" is kept because dropping it would remove real customers, CLIENTNUM is an identiier not a measure

In [29]:
# Create a cleaned copy of the original dataset
df_clean = df.copy()

# Keep "Unknown" as its own category
# Remove "$" so Jupyter displays income labels correctly
df_clean["Income_Category"] = df_clean["Income_Category"].str.replace("$", "", regex=False)

# Drop identifier and the dataset author's model outputs
nb_cols = [c for c in df_clean.columns if c.startswith("Naive_Bayes")]
df_clean = df_clean.drop(columns=["CLIENTNUM"] + nb_cols)
print(df_clean.shape)

(10127, 20)


In [31]:
# Create customer age groups, transacion activity and credit utiilisation groups
df_clean["Age_Group"] = pd.cut(df_clean["Customer_Age"], bins=[0, 30, 40, 50, 60, 120],
                               labels=["30 and below", "31-40", "41-50", "51-60", "Above 60"])
df_clean["Transaction_Activity"] = pd.cut(df_clean["Total_Trans_Ct"], bins=[0, 40, 70, np.inf],
                                          labels=["Low", "Medium", "High"])
df_clean["Utilisation_Group"] = pd.cut(df_clean["Avg_Utilization_Ratio"], bins=[-0.01, 0.20, 0.50, 1.0],
                                       labels=["Low", "Moderate", "High"])

# Order columns so Power BI sorts groups correctly
df_clean["Age_Order"] = df_clean["Age_Group"].cat.codes + 1
df_clean["Activity_Order"] = df_clean["Transaction_Activity"].cat.codes + 1
df_clean["Utilisation_Order"] = df_clean["Utilisation_Group"].cat.codes + 1

df_clean[["Customer_Age", "Age_Group", "Total_Trans_Ct", "Transaction_Activity",
          "Avg_Utilization_Ratio", "Utilisation_Group"]].head(10)

,Customer_Age,Age_Group,Total_Trans_Ct,Transaction_Activity,Avg_Utilization_Ratio,Utilisation_Group
0,45,41-50,42,Medium,0.061,Low
1,49,41-50,33,Low,0.105,Low
2,51,51-60,20,Low,0.000,Low
3,40,31-40,20,Low,0.760,High
4,40,31-40,28,Low,0.000,Low
5,44,41-50,24,Low,0.311,Moderate
6,51,51-60,31,Low,0.066,Low
7,32,31-40,36,Low,0.048,Low
8,37,31-40,24,Low,0.113,Low
9,48,41-50,32,Low,0.144,Low


In [35]:
print(df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
for c in ["Age_Group", "Transaction_Activity", "Utilisation_Group"]:
    print(df_clean[c].value_counts().sort_index(), "\n")

df_clean.to_csv(r"C:\Users\USER\Documents\MIT8339_BI_Virtual_Lab\data\BankChurners.csv", index=False)

(10127, 26)
Missing values: 0
Duplicate rows: 0
Age_Group
30 and below     265
31-40           2132
41-50           4652
51-60           2673
Above 60         405
Name: count, dtype: int64 

Transaction_Activity
Low       1938
Medium    3750
High      4439
Name: count, dtype: int64 

Utilisation_Group
Low         5346
Moderate    2236
High        2545
Name: count, dtype: int64 



In [37]:
import os
print (os.getcwd())

C:\Users\USER
